In [87]:
import pandas as pd
import numpy as np
from numpy import linalg

In [88]:
df = pd.read_csv('https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/car_fuel_efficiency_2026.csv')
df.columns = df.columns.str.lower().str.replace(' ', '_')
df = df[['engine_displacement','horsepower','vehicle_weight','model_year','fuel_efficiency_mpg']]
y = df.fuel_efficiency_mpg.values
df = df.drop('fuel_efficiency_mpg', axis=1)
#df.head(3)

In [89]:
#Q1
df.isnull().sum()

engine_displacement      0
horsepower             877
vehicle_weight           0
model_year               0
dtype: int64

In [90]:
#Q2
df.horsepower.median()

np.float64(254.0)

In [91]:
def prepare_X(df):
    df_num = df.copy()
    df_num = df_num.fillna(0)
    X = df_num.values
    return X

def train_linear_regression(X, y):
    ones = np.ones(X.shape[0])
    X = np.column_stack([ones, X])

    XTX = X.T.dot(X)
    XTX_inv = np.linalg.inv(XTX)
    w = XTX_inv.dot(X.T).dot(y)
    
    return w[0], w[1:]

def rmse(y, y_pred):
    error = y_pred - y
    mse = (error ** 2).mean()
    return np.sqrt(mse)

In [92]:
n = len(df)
n_val = int(n * 0.2)
n_test = int(n * 0.2)
n_train = n - n_val - n_test

np.random.seed(42)
idx = np.arange(n)
np.random.shuffle(idx)

df_train = df.iloc[idx[:n_train]]
df_val = df.iloc[idx[n_train:n_train + n_val]]
df_test = df.iloc[idx[n_train + n_val:]]

y_train = y[idx[:n_train]]
y_val = y[idx[n_train:n_train + n_val]]
y_test = y[idx[n_train + n_val:]]

In [93]:
#Q3
#Option zeros
df_train_zeros = df_train.fillna(0)
df_val_zeros = df_val.fillna(0)
X_train_zeros = prepare_X(df_train_zeros)
w_0_zeros, w_zeros = train_linear_regression(X_train_zeros, y_train)
X_val_zeros = prepare_X(df_val_zeros)
y_pred_zeros = w_0_zeros + X_val_zeros.dot(w_zeros)
print('validation zeros:', rmse(y_val, y_pred_zeros))

validation zeros: 2.20529319475341


In [94]:
#Q3
#Option mean
df_train_mean = df_train.fillna(df_train.horsepower.mean())
df_val_mean = df_val.fillna(df_train.horsepower.mean())
X_train_mean = prepare_X(df_train_mean)
w_0_mean, w_mean = train_linear_regression(X_train_mean, y_train)
X_val_mean = prepare_X(df_val_mean)
y_pred_mean = w_0_mean + X_val_mean.dot(w_mean)
print('validation mean:', rmse(y_val, y_pred_mean))

validation mean: 2.2018174845688243


In [95]:
print(round(rmse(y_val, y_pred_zeros), 3))
print(round(rmse(y_val, y_pred_mean), 3))

2.205
2.202


In [96]:
def train_linear_regression_reg(X, y, r=0.0):
    ones = np.ones(X.shape[0])
    X = np.column_stack([ones, X])

    XTX = X.T.dot(X)
    reg = r * np.eye(XTX.shape[0])
    XTX = XTX + reg

    XTX_inv = np.linalg.inv(XTX)
    w = XTX_inv.dot(X.T).dot(y)
    
    return w[0], w[1:]

In [97]:
#Q4
df_train = df_train.fillna(0)
df_val = df_val.fillna(0)
X_train = prepare_X(df_train)
X_val = prepare_X(df_val)
r_try = [0, 0.01, 0.1, 1, 5, 10, 100]
for r in r_try:
    w_0, w = train_linear_regression_reg(X_train, y_train, r=r)
    y_pred = w_0 + X_val.dot(w)
    score = rmse(y_val, y_pred)
    print(r, round(score, 4))

0 2.2053
0.01 2.2058
0.1 2.2241
1 2.3492
5 2.4094
10 2.4195
100 2.4292


In [99]:
#Q5
rmses = []
for seed in [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]:
    n = len(df)
    n_val = int(n * 0.2)
    n_test = int(n * 0.2)
    n_train = n - n_val - n_test

    np.random.seed(seed)
    idx = np.arange(n)
    np.random.shuffle(idx)

    df_train = df.iloc[idx[:n_train]]
    df_val = df.iloc[idx[n_train:n_train + n_val]]
    df_test = df.iloc[idx[n_train + n_val:]]

    y_train = y[idx[:n_train]]
    y_val = y[idx[n_train:n_train + n_val]]
    y_test = y[idx[n_train + n_val:]]

    df_train = df_train.fillna(0)
    df_val = df_val.fillna(0)
    X_train = prepare_X(df_train)
    X_val = prepare_X(df_val)

    w_0, w = train_linear_regression(X_train, y_train)
    y_pred = w_0 + X_val.dot(w)

    score = rmse(y_val, y_pred)
    rmses.append(score)
    print(seed, round(score, 3))

std = np.std(rmses)
print('std:', round(std, 3))

0 2.239
1 2.202
2 2.163
3 2.204
4 2.202
5 2.246
6 2.27
7 2.191
8 2.217
9 2.207
std: 0.029


In [101]:
#Q6
n = len(df)
n_val = int(n * 0.2)
n_test = int(n * 0.2)
n_train = n - n_val - n_test

np.random.seed(9)
idx = np.arange(n)
np.random.shuffle(idx)

df_train = df.iloc[idx[:n_train]]
df_val = df.iloc[idx[n_train:n_train + n_val]]
df_test = df.iloc[idx[n_train + n_val:]]

y_train = y[idx[:n_train]]
y_val = y[idx[n_train:n_train + n_val]]
y_test = y[idx[n_train + n_val:]]

df_full_train = pd.concat([df_train, df_val])
df_full_train = df_full_train.reset_index(drop=True)
df_full_train = df_full_train.fillna(0)
df_test = df_test.fillna(0)

X_full_train = prepare_X(df_full_train)
y_full_train = np.concatenate([y_train, y_val])
w0, w = train_linear_regression_reg(X_full_train, y_full_train, r=0.001)

X_test = prepare_X(df_test)
y_pred = w0 + X_test.dot(w)
score = rmse(y_test, y_pred)
score

np.float64(2.235827747191731)